# Computing a social cost of carbon with dscim-cil

Everything below runs on small generated inputs; no external data is
needed. The `run` extra must be installed
(`uv pip install ".[run]"`) so dscim itself is available.

## Inputs and config

Small versions of every input a run needs are generated here: an
economics zarr, reduced-damage zarrs, a GMST table, FaIR temperature
projections, and a pulse conversion file. The config below points at them and is saved as
`demo_data/demo.yml`.

In [1]:
import pathlib
import sys

import yaml

repo = pathlib.Path.cwd().resolve()
if not (repo / "tests").exists():
    repo = repo.parent
sys.path.insert(0, str(repo / "tests"))
import fixture_factory

data = repo / "examples" / "demo_data"
data.mkdir(exist_ok=True)
config = fixture_factory.ssp_fixture_config(data)
config_path = data / "demo.yml"
config_path.write_text(yaml.safe_dump(config))
print(f"wrote {config_path}")

/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/demo.yml


## The pipeline

`stages` explains the pipeline without touching any data: each stage,
what it reads and writes, and which dimensions it collapses.

In [2]:
!dscim-cil stages

pipeline
├── 1. sum-sectors  (ssp)
│   ├── Sum member sectors' delta and histclim into an aggregate sector zarr 
│   │   (e.g. AMEL from agriculture+mortality+energy+labor).
│   ├── wraps dscim.preprocessing.preprocessing.sum_AMEL
│   ├── consumes
│   │   └── member sector damages zarrs (sectors.<name>.sector_path)
│   ├── produces
│   │   └── aggregate sector damages zarr (its sector_path)
│   └── dimensions
│       └── no collapse: sums variables across sectors at identical dimensions 
│           (preprocessing.py sum_AMEL)
├── 2. reduce  (ssp)
│   ├── Collapse the batch dimension of sector damages against socioeconomics, 
│   │   producing the reduced-damage zarrs runs consume.
│   ├── wraps dscim.preprocessing.preprocessing.reduce_damages
│   ├── consumes
│   │   ├── sector damages zarrs (batch chunked at 15)
│   │   └── econ zarr (external)
│   ├── produces
│   │   └── reduced zarrs: adding_up_{cc,no_cc}.zarr and 
│   │       risk_aversion_{cc,no_cc}_eta{eta}.zarr per sector
│   

## Options

`options` lists every option dscim accepts. Each carries a status:
`supported`, `unsupported` (accepted by dscim but broken or restricted,
with the reason), `dead` (accepted and ignored), or `removed` (only on
other dscim branches).

In [3]:
!dscim-cil options --status unsupported

option                      status       stages  default                        
ecs_mask_name               unsupported  fair    None                           
full_uncertainty_quantiles  unsupported  output  (0.01, 0.05, 0.17, 0.25, 0.5,  
                                                 0.75, 0.83, 0.95, 0.99)        
quantreg                    unsupported  reduce  False                          
quantreg_quantiles          unsupported  fit     (0.05, 0.1, 0.15, 0.2, 0.25,   
                                                 0.3, 0.35, 0.4, 0.45, 0.5,     
                                                 0.55, 0.6, 0.65, 0.7, 0.75,    
                                                 0.8, 0.85, 0.9, 0.95)          


`explain` gives the full record for one option, including why a
value is unsupported and where in dscim's source that behavior lives:

In [4]:
!dscim-cil explain discounting_type constant_gwr

discounting_type: Discounting scheme; also controls damage-function fit grouping and population collapse.
  status: supported
  stages: fit, discount
  modes: ssp, rff
  required in config; dscim would default to None if unset, but dscim-cil never applies that silently
  source: main_recipe.py:88 (default None); accepted set main_recipe.py:52-61; assert main_recipe.py:245-247
  values:
    'constant_gwr': unsupported [library]
        Listed in DISCOUNT_TYPES but its non-constant discount-factor path has no branch in calculate_stream_discount_factors and raises UnboundLocalError; never used in any production repo or in dscim's own test matrix.
        source: main_recipe.py:52-61; calculate_stream_discount_factors has no constant_gwr branch; dscim's tests/conftest.py discount_types fixture omits it


## Constraints

Some restrictions span options. `constraints` lists them with their
source citations; what it prints is what `validate` enforces.

In [5]:
!dscim-cil constraints

rule                           applies            description                   
median-params-fair-dims        requires; ssp,rff  fair_aggregation containing   
                                                  median_params requires        
                                                  fair_dims to be exactly       
                                                  [simulation]; dscim documents 
                                                  this and does not enforce it. 
                                                  main_recipe.py:47 (docstring);
                                                  no matching assert in __init__
clip-gmsl-formula              requires; ssp,rff  clip_gmsl requires the formula
                                                  to be one of the two          
                                                  gmsl-quadratic formulas.      
                                                  main_recipe.py:264-270        
                            

## Validation

`validate` reports every problem at once.

In [6]:
!dscim-cil validate {config_path}

config is valid


## The plan

`plan` lays out the pipeline for this config as ordered steps, each
marked ready or blocked, with the command that produces every missing
input.

In [7]:
!dscim-cil plan {config_path}

root: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data
1. run: fit and integrate labor  [ready]
     in  [ok] fair.nc
     in  [ok] conversion.nc
     in  [ok] econ.zarr
     in  [ok] gmst.csv
     in  [ok] reduced/labor/adding_up_cc.zarr  <- dscim-cil reduce
     in  [ok] reduced/labor/adding_up_no_cc.zarr  <- dscim-cil reduce
     out [new] results/labor/2020/unm…ta2.0_rho0.0001_scc.nc4
     out [new] results/labor/2020/unm…01_uncollapsed_sccs.nc4


A dry run first prints the settings the sweep will use, marking
which values came from the config and which are dscim defaults (the
result-selecting values must always be explicit), then summarizes the
expanded runs:

In [8]:
!dscim-cil run {config_path} --dry-run

settings                                     
option                value           origin 
discounting_type      euler_ramsey    config 
discrete_discounting  False           default
eta                   2.0             config 
ext_method            global_c_ratio  default
fair_aggregation      ['ce', 'mean']  config 
fit_type              ols             default
gases                 ['CO2_Fossil']  config 
pulse_year            2020            config 
recipe                adding_up       config 
rho                   0.0001          config 
sector                labor           config 
weitzman_parameter    [0.1]           config 
mode: ssp   runs: 1  (1 sectors x 1 pulse_years x 1 menu pairs x 1 eta_rho x 1 
masks x 1 fair_dims)
missing inputs: none
outputs: 2 files, 0 already exist
blocked runs: 0 of 1  (missing inputs)
use --verbose or --runs N[,N...] for per-run detail


## Running

One real run: fit the damage function on the generated damages, apply
it to the FaIR projections, discount, and write SCC files. On inputs
this small it takes a few seconds.

In [9]:
!dscim-cil run {config_path}

settings                                     
option                value           origin 
discounting_type      euler_ramsey    config 
discrete_discounting  False           default
eta                   2.0             config 
ext_method            global_c_ratio  default
fair_aggregation      ['ce', 'mean']  config 
fit_type              ols             default
gases                 ['CO2_Fossil']  config 
pulse_year            2020            config 
recipe                adding_up       config 
rho                   0.0001          config 
sector                labor           config 
weitzman_parameter    [0.1]           config 


INFO running labor 2020 adding_up/euler_ramsey eta=2.0 rho=0.0001

 Executing 
        Running adding_up
        sector: labor
        discounting: euler_ramsey
        eta: 2.0
        rho: 0.0001
        
INFO 
 Executing 
        Running adding_up
        sector: labor
        discounting: euler_ramsey
        eta: 2.0
        rho: 0.0001
        
Processing damage functions ...
INFO Processing damage functions ...
Existing damage functions not found. Damage points will be loaded.
INFO Existing damage functions not found. Damage points will be loaded.
Adding up aggregated damages found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/reduced/labor/adding_up_cc.zarr, /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/reduced/labor/adding_up_no_cc.zarr. These are being loaded...
INFO Adding up aggregated damages found at /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/reduced/labor/adding_up_cc.zarr, /project

Extrapolating global consumption.
INFO Extrapolating global consumption.
End-of-century growth rates are not capped.


/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in RankOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
/project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/.venv/lib/python3.13/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placem

Processing SCC calculation ...
INFO Processing SCC calculation ...


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked/adding_up_euler_ramsey_eta2.0_rho0.0001_scc.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked/adding_up_euler_ramsey_eta2.0_rho0.0001_scc.nc4


Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked/adding_up_euler_ramsey_eta2.0_rho0.0001_uncollapsed_sccs.nc4
INFO Saving /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked/adding_up_euler_ramsey_eta2.0_rho0.0001_uncollapsed_sccs.nc4


Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked
INFO Results available: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked


completed: labor 2020 adding_up/euler_ramsey eta=2.0 rho=0.0001 (metadata: /project/cil/home_dirs/scadavidsanchez/repos/dscim-cil/examples/demo_data/results/labor/2020/unmasked/adding_up_euler_ramsey_eta2.0_rho0.0001_run_metadata.yaml)


## Results

Every run writes its artifacts plus a `*_run_metadata.yaml` recording
the resolved settings with their provenance, the dscim version and
commit that produced the result, and dependency versions.

In [10]:
import xarray as xr

results = data / "results" / "labor" / "2020" / "unmasked"
stem = "adding_up_euler_ramsey_eta2.0_rho0.0001"
scc = xr.open_dataset(results / f"{stem}_scc.nc4")
print(scc)

<xarray.Dataset> Size: 348B
Dimensions:             (fair_aggregation: 2, weitzman_parameter: 1,
                         discount_type: 1, ssp: 2, model: 2, rcp: 2, gas: 1)
Coordinates:
  * fair_aggregation    (fair_aggregation) <U4 32B 'ce' 'mean'
  * weitzman_parameter  (weitzman_parameter) <U3 12B '0.1'
  * discount_type       (discount_type) <U12 48B 'euler_ramsey'
  * ssp                 (ssp) <U4 32B 'SSP2' 'SSP3'
  * model               (model) <U2 16B 'm1' 'm2'
  * rcp                 (rcp) <U5 40B 'rcp45' 'rcp85'
  * gas                 (gas) <U10 40B 'CO2_Fossil'
Data variables:
    scc                 (fair_aggregation, weitzman_parameter, discount_type, ssp, model, rcp, gas) float64 128B ...
Attributes: (12/49)
    sector_path:                    None
    save_path:                      /project/cil/home_dirs/scadavidsanchez/re...
    gdppc_bottom_code:              39.39265060424805
    subset_dict:                    {'ssp': ['SSP2', 'SSP3']}
    econ_vars:              

In [11]:
metadata = yaml.safe_load((results / f"{stem}_run_metadata.yaml").read_text())
print("dscim:", metadata["dscim_version"], "commit", metadata["dscim_commit"])
print("eta came from:", metadata["provenance"]["eta"])
print("ext_method came from:", metadata["provenance"]["ext_method"])

dscim: 0.7.1.dev31+g6a1f4d7e5 commit 6a1f4d7e5
eta came from: config
ext_method came from: default
